# Institution Resolver v3 — Gemma4:e4b Hakem (Judge) Kaggle Notebook — TAM DOSYA, A→Z

2026-08-13, canlı denemede bulunup düzeltilen TÜM sorunlar bu sürüme işlendi.
Baştan sona bu sırayla giderseniz aynı hatalara tekrar takılmamalısınız.

## ⚠️ ZORUNLU ön koşullar (atlarsanız ilerleyen hücreler patlar)

1. **Accelerator = GPU T4 x2 olmalı, P100 DEĞİL.** P100 (Pascal, sm_60), güncel
   PyTorch'un artık derlenmiş çekirdek desteği vermediği bir mimari - embedding
   adımı (`sentence-transformers`) `AcceleratorError: no kernel image` ile çöker.
   Ollama'nın kendisi bu sorunu yaşamaz (ayrı CUDA çekirdekleri kullanır) ama
   indeksleme adımı PyTorch gerektiriyor. Sağ panel → Session options →
   Accelerator → **GPU T4 x2** seçin.
2. **Internet açık VE hesap telefonla doğrulanmış olmalı.** Notebook
   Settings'te "Internet" anahtarı kapalıysa `Permission
   'kernelSessions.enableInternet' was denied` hatası alırsınız - kaggle.com →
   Settings → Phone Verification'ı tamamlayın, sonra anahtarı açın.
3. **DÖRT Kaggle Dataset'i hazır ve "Add Input" ile eklenmiş olmalı:**
   - `gemma4-e4b-ollama` — model dosyaları
   - `needs-review-full` — işlenecek sorgular (TAM dosya, 143.039 satır, A→Z)
   - `institution-catalog-canonical` — **kurum kataloğunun kendisi**
     (`parent_canonical.jsonl` + `subunit_canonical.jsonl`) - bunsuz
     Elasticsearch'e indekslenecek hiçbir kurum verisi olmaz, `index --embeddings`
     `FileNotFoundError` verir.
   - `kaggle-judge-output` — boş, ara yedeklerin push edileceği hedef

   **Kaggle'ın gerçek mount yolu** `/kaggle/input/<slug>` DEĞİL,
   `/kaggle/input/datasets/<kullanici_adi>/<slug>` şeklinde çıkıyor (2026-08-13
   canlı doğrulama) - 2. hücredeki yollar buna göre ayarlı.

## 🔁 Oturum yeniden başlarsa (Accelerator/Internet değişince kaçınılmaz)

Aşağıdaki hücreleri **sırayla baştan** çalıştırın - hiçbiri "bir öncekinde
kaldığım yerden devam eder" varsaymaz, hepsi idempotent (tekrar çalıştırmak
güvenli):
1) GPU kontrolü → 2) İnternet kontrolü → 3) Yollar → 4) Git clone → 5) Katalog
kopyalama → 6) ES kurulumu → 7) ES sağlık kontrolü → 8) Ollama+model →
9) Python paket kurulumu → 10) ES indeksleme

`pip install` ve `setup-es`/`index` hücreleri artık kendi içlerinde
`%cd {REPO_DIR}` çalıştırıyor - yanlış dizinde kalmış olsanız bile düzeltir.

## Genel mimari

`needs_review_subset.csv`'yi (TAM, 143.039 satır) A'dan başlayarak işliyoruz.
Colab (ayrı, dokunulmuyor) aynı dosyayı Z'den işliyor - iki taraf ortada
buluşana kadar devam eder, örtüşen kısım final birleştirmede (query metnine
göre tekilleştirme) zararsızca elenir. Yerelde daha önce işlenen 1.002 satır
da kaybolmadı, birleştirmeye dahil edilecek.

## Kalıcılık ve GPU kotası

`/kaggle/working` oturum bitince SIFIRLANIR. Bunun yerine periyodik olarak
Kaggle API ile çıktıyı bir Dataset'e push ediyoruz (bkz. "Ara Yedekleme"
bölümü). Token için: kaggle.com → Settings → API → "Create New Token" →
`KGAT_...` ile başlayan token'ı kopyalayın → bu notebook'ta Add-ons → Secrets
→ isim `KAGGLE_API_TOKEN`, değer token'ın kendisi. **Token'ı hiçbir yere
(sohbet, kod, commit) açık metin yazmayın.**

Ücretsiz Kaggle hesabı haftada ~30 saat GPU süresi veriyor, tekli oturum
genelde ~9-12 saatte kesiliyor - tam dosya (143.039 satır) tek oturumda
bitmez, döngü `--resume` ile kaldığı yerden devam eder.

## 1) Donanım & GPU Kontrolü

In [ ]:
!nvidia-smi

import subprocess
gpu_name = subprocess.run(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    capture_output=True, text=True,
).stdout.strip()
print("Tespit edilen GPU:", gpu_name or "(YOK - Accelerator None mi secili?)")
if "P100" in gpu_name:
    raise RuntimeError(
        "P100 secili! Bu, PyTorch'un artik desteklemedigi bir mimari (sm_60) - "
        "embedding adimi CUDA hatasiyla cokecek. Sag panel -> Session options -> "
        "Accelerator -> 'GPU T4 x2' secip oturumu yeniden baslatin."
    )
if not gpu_name:
    raise RuntimeError(
        "Hic GPU bulunamadi. Sag panel -> Session options -> Accelerator -> "
        "'GPU T4 x2' secip oturumu yeniden baslatin."
    )
print("GPU uygun, devam edilebilir.")

## 1.5) İnternet Bağlantı Kontrolü

Bunu atlamayın - internet kapalıyken sonraki hücreler (git clone, pip
install, ES/Ollama indirme) `Could not resolve host` gibi belirsiz hatalarla
uzun uzun uğraştırır.

In [ ]:
import os

# Bilgiler dolduruldu (2026-08-13) - dört dataset de hazır:
#   https://www.kaggle.com/datasets/mcangultekin/gemma4-e4b-ollama
#   https://www.kaggle.com/datasets/mcangultekin/needs-review-full
#   https://www.kaggle.com/datasets/mcangultekin/institution-catalog-canonical
#   https://www.kaggle.com/datasets/mcangultekin/kaggle-judge-output
# NOT: Kaggle'ın gercek mount yolu "/kaggle/input/<slug>" DEGIL,
# "/kaggle/input/datasets/<kullanici_adi>/<slug>" seklinde cikti (2026-08-13
# canli dogrulama - eski varsayimdan farkli, dikkat).
KAGGLE_USERNAME = "mcangultekin"
KAGGLE_MODEL_DATASET_PATH = "/kaggle/input/datasets/mcangultekin/gemma4-e4b-ollama"
KAGGLE_INPUT_CSV_PATH = "/kaggle/input/datasets/mcangultekin/needs-review-full/needs_review_subset.csv"
KAGGLE_CATALOG_DATASET_PATH = "/kaggle/input/datasets/mcangultekin/institution-catalog-canonical"
KAGGLE_OUTPUT_DATASET_SLUG = "kaggle-judge-output"

assert os.path.isdir(KAGGLE_MODEL_DATASET_PATH), f"Model dataset bulunamadı: {KAGGLE_MODEL_DATASET_PATH} - 'Add Input' ile eklediniz mi?"
assert os.path.isfile(KAGGLE_INPUT_CSV_PATH), f"Girdi CSV bulunamadı: {KAGGLE_INPUT_CSV_PATH}"
assert os.path.isdir(KAGGLE_CATALOG_DATASET_PATH), f"Katalog dataset bulunamadı: {KAGGLE_CATALOG_DATASET_PATH} - 'Add Input' ile eklediniz mi?"
print("Yollar doğrulandı.")

## 3.5) Kurum Kataloğunu Kopyala

Elasticsearch'e indekslenecek asıl veri - Kaggle input dataset'i salt-okunur
olduğu için repo'nun beklediği `data/processed/` yoluna kopyalıyoruz.

In [ ]:
import shutil, os

REPO_DIR = "/kaggle/working/institution_resolver_v3"  # sabit yol - baska hucreye bagimli degil
os.chdir(REPO_DIR)  # savunma: onceki hucre calismamis/oturum yenilenmis olsa bile
os.makedirs("data/processed", exist_ok=True)
shutil.copy(f"{KAGGLE_CATALOG_DATASET_PATH}/parent_canonical.jsonl", "data/processed/parent_canonical.jsonl")
shutil.copy(f"{KAGGLE_CATALOG_DATASET_PATH}/subunit_canonical.jsonl", "data/processed/subunit_canonical.jsonl")
print("Katalog dosyaları kopyalandı:", os.listdir("data/processed"))

## 3) Koda Erişim (Git Clone / Pull)

In [ ]:
REPO_DIR = "/kaggle/working/institution_resolver_v3"
BRANCH = "feat/gate-asama1"

import os
if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} https://github.com/mcangultekin/institution_resolver_v3.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch origin && git checkout {BRANCH} && git pull
%cd {REPO_DIR}

## 4) Elasticsearch Kurulumu ve Başlatma

In [ ]:
%%bash
set -e
ES_VERSION=8.14.0
if [ ! -d /kaggle/working/es ]; then
  wget -q https://artifacts.elastic.co/downloads/elasticsearch/elasticsearch-${ES_VERSION}-linux-x86_64.tar.gz -O /kaggle/working/es.tar.gz
  mkdir -p /kaggle/working/es
  tar -xzf /kaggle/working/es.tar.gz -C /kaggle/working/es --strip-components=1
fi

grep -q '^discovery.type' /kaggle/working/es/config/elasticsearch.yml || cat >> /kaggle/working/es/config/elasticsearch.yml <<EOF
discovery.type: single-node
xpack.security.enabled: false
xpack.security.http.ssl.enabled: false
xpack.ml.enabled: false
EOF

cat > /kaggle/working/es/config/elasticsearch.policy <<'POLEOF'
grant {
  permission java.io.FilePermission "/sys/-", "read";
  permission java.io.FilePermission "/proc/-", "read";
};
POLEOF

sysctl -w vm.max_map_count=262144 || true
id -u esuser &>/dev/null || useradd -m esuser
chown -R esuser:esuser /kaggle/working/es

pkill -f 'org.elasticsearch.bootstrap.Elasticsearch' 2>/dev/null || true
sleep 1
sudo -u esuser env ES_JAVA_OPTS="-Xms4g -Xmx4g -Djava.security.policy=/kaggle/working/es/config/elasticsearch.policy" \
  setsid /kaggle/working/es/bin/elasticsearch < /dev/null > /kaggle/working/es/es.log 2>&1 &
disown
sleep 3

In [ ]:
import time, requests

for _ in range(60):
    try:
        r = requests.get("http://localhost:9200/_cluster/health", timeout=2)
        if r.status_code == 200:
            print("Elasticsearch Sağlıklı:", r.json())
            break
    except Exception:
        pass
    time.sleep(2)
else:
    raise RuntimeError("ES başlatılamadı, /kaggle/working/es/es.log kontrol edin")

In [ ]:
import os
REPO_DIR = "/kaggle/working/institution_resolver_v3"  # sabit yol - baska hucreye bagimli degil
os.chdir(REPO_DIR)  # savunma: cwd kaymis olabilir (oturum yenilenmesi vb.)
print("cwd:", os.getcwd())

!pip uninstall -y torchaudio torchvision 2>/dev/null || true
!pip install -q --force-reinstall -e ".[dev,embed,llm,api]"

In [ ]:
%%bash
set -e
apt-get update -qq && apt-get install -y -qq zstd

if ! command -v ollama &> /dev/null; then
  curl -fsSL https://ollama.com/download/ollama-linux-amd64.tar.zst | tar --zstd -x -C /usr
fi

ollama --version

## 5.5) Ollama Sunucusunu Başlat + Modeli Bağla

Önceki hücre sadece `ollama` CLI'sini kurdu - sunucu ayrı başlatılmalı.
`OLLAMA_MODELS`'i doğrudan salt-okunur Kaggle dataset'ine yönlendiriyoruz,
`ollama pull` gerekmiyor.

In [ ]:
import os, subprocess, time, requests

# OLLAMA_MODELS'i dogrudan salt-okunur Kaggle dataset'ine (KAGGLE_MODEL_DATASET_PATH)
# isaret ediyoruz - "ollama pull" gerekmiyor, model dosyalari zaten dataset icinde
# dogru dizin yapisinda (manifests/... + blobs/...) hazir.
os.environ["OLLAMA_MODELS"] = KAGGLE_MODEL_DATASET_PATH
os.environ["OLLAMA_KEEP_ALIVE"] = "-1"

subprocess.Popen(
    ["ollama", "serve"],
    stdout=open("/kaggle/working/ollama.log", "w"),
    stderr=subprocess.STDOUT,
)

for _ in range(30):
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=2)
        if r.status_code == 200:
            names = [m["name"] for m in r.json().get("models", [])]
            print("Ollama sağlıklı, görünen modeller:", names)
            assert any("gemma4" in n for n in names), "gemma4:e4b listede yok - OLLAMA_MODELS yolu/dataset yapisini kontrol edin"
            break
    except requests.exceptions.ConnectionError:
        pass
    time.sleep(1)
else:
    raise RuntimeError("Ollama başlamadı, /kaggle/working/ollama.log kontrol edin")

In [ ]:
import os
REPO_DIR = "/kaggle/working/institution_resolver_v3"  # sabit yol - baska hucreye bagimli degil
os.chdir(REPO_DIR)  # savunma: cwd kaymis olabilir
assert os.path.isfile("data/processed/parent_canonical.jsonl"), "Katalog dosyalari yok - 3.5 hucreyi calistirdiniz mi?"

!python3 -m institution_resolver_v3.cli.main setup-es
!python3 -m institution_resolver_v3.cli.main index --embeddings

## 7.5) Tekli Sorgu Testi (Her Şey Çalışıyor mu?)

In [ ]:
!python3 -m institution_resolver_v3.cli.main judge "Milli Egitim Bakanligi" --model "gemma4:e4b"

## 8) Ara Yedekleme Fonksiyonu — Kaggle API ile çıktı Dataset'ine push

`/kaggle/working` oturum bitince silinir - bu yüzden Colab'deki "Drive'a kopyala"
yerine burada "Kaggle Dataset'i olarak yükle (version)" kullanıyoruz. Kaggle'ın
yeni API akışı `KGAT_...` ile başlayan TEK bir token kullanıyor (eski
kaggle.json/username+key akışı değil) - token'ı Secrets'tan (`KAGGLE_API_TOKEN`)
okuyup `~/.kaggle/access_token` dosyasına yazıyoruz (Kaggle CLI'nin beklediği yol).

In [ ]:
import os, json, subprocess

def _setup_kaggle_api():
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    token = secrets.get_secret("KAGGLE_API_TOKEN")
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    with open(os.path.expanduser("~/.kaggle/access_token"), "w") as f:
        f.write(token.strip())
    os.chmod(os.path.expanduser("~/.kaggle/access_token"), 0o600)
    print("Kaggle API token hazır.")

_setup_kaggle_api()

def backup_to_kaggle_dataset(local_csv_path, dataset_slug=KAGGLE_OUTPUT_DATASET_SLUG, message=None):
    """local_csv_path'i, onceden olusturulmus bos bir Kaggle Dataset'inin yeni
    versiyonu olarak push eder. Dataset ilk kez bu notebook'tan olusturulmuyor -
    kaggle.com uzerinde bos bir dataset onceden elle olusturulmus olmali (bkz. ust not)."""
    import shutil, tempfile, time as _time
    staging = tempfile.mkdtemp()
    shutil.copy(local_csv_path, os.path.join(staging, os.path.basename(local_csv_path)))
    meta = {
        "title": dataset_slug,
        "id": f"{KAGGLE_USERNAME}/{dataset_slug}",
        "licenses": [{"name": "CC0-1.0"}],
    }
    with open(os.path.join(staging, "dataset-metadata.json"), "w") as f:
        json.dump(meta, f)
    msg = message or f"auto-backup {_time.strftime('%Y-%m-%d %H:%M:%S')}"
    result = subprocess.run(
        ["kaggle", "datasets", "version", "-p", staging, "-m", msg, "-r", "zip"],
        capture_output=True, text=True,
    )
    print(result.stdout[-500:] if result.stdout else "")
    if result.returncode != 0:
        print("UYARI: yedekleme başarısız:", result.stderr[-500:])
    else:
        print("Yedeklendi ->", dataset_slug)
    shutil.rmtree(staging, ignore_errors=True)

## 9) İşçi (`--workers`) Sayısı Testi

2026-08-12 ölçümü (yerelde): 8 işçi ÇÖKTÜ, 6 işçide güvenilirlik düştü, workers=2
en hızlı VE en güvenilir çıktı; workers=1 tek makinede çok yavaş kaldı (bu yüzden
yerel taraf devre dışı bırakıldı). Kaggle'ın kendi donanımında doğrulamak için
bu hücre var - GPU'lu ortamda daha yüksek worker sayıları güvenli olabilir.

In [ ]:
import os, csv, time

WORKER_COUNTS = [1, 2, 4, 8, 16]
TEST_LIMIT = 30
MODEL_TAG = "gemma4:e4b"

print(f"GEMMA4:E4B ISCI HIZ TESTI (hakem dahil, kaggle diliminin ilk {TEST_LIMIT} satiri)...\n")

results = []
for workers in WORKER_COUNTS:
    out = f"/kaggle/working/worker_test_w{workers}.csv"
    if os.path.exists(out):
        os.remove(out)

    t0 = time.time()
    !python3 -m institution_resolver_v3.cli.main inventory-batch "{KAGGLE_INPUT_CSV_PATH}" --model "{MODEL_TAG}" --out "{out}" --limit {TEST_LIMIT} --workers {workers} > /kaggle/working/worker_test_w{workers}.log 2>&1
    dt = time.time() - t0

    with open(out, newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    ok = sum(1 for r in rows if r.get("status") == "ok")
    qps = TEST_LIMIT / dt if dt > 0 else 0
    results.append((workers, dt, dt / TEST_LIMIT, qps, ok))
    print(f"workers={workers:>2d} -> toplam={dt:6.1f}s | {dt/TEST_LIMIT:5.2f} sn/sorgu | {qps:5.2f} sorgu/sn | basarili={ok}/{TEST_LIMIT}")

print("\n--- OZET (GEMMA4:E4B, KAGGLE DILIMI) ---")
best = min(results, key=lambda r: r[2])
print(f"En hizli: workers={best[0]} -> {best[2]:.2f} sn/sorgu")
print(f"47.203 satir (bu Kaggle'in payi) icin bu workers ile tahmini sure: {47203 * best[2] / 3600:.1f} saat")
print("\nNot: cokme/basarisizlik gorulen worker sayilarini bir sonraki hucrede KULLANMAYIN.")

## 10) TAM KOŞU: Tam Dosya (A→Z) + gemma4:e4b Hakem

**Kesinti riski Colab'dan farklı:** Kaggle oturumu (GPU kotası dolduğunda ya da
9-12 saat sınırına ulaşınca) OTOMATİK olarak sona erer, `/kaggle/working` silinir.
Bu yüzden burada batch'i tek seferde `--resume` ile sonuna kadar çalıştırmak yerine,
**periyodik olarak duraklat + yedekle** döngüsü kullanıyoruz - her `CHUNK` sorguda bir
Kaggle Dataset'ine push ediyor, böylece oturum aniden kesilse bile en fazla bir
`CHUNK`'lık iş kaybedersiniz.

Colab aynı anda TERS sırada (Z→A) aynı tam dosyayı işliyor - iki taraf ortada
buluşana kadar devam eder, örtüşen kısım final birleştirmede (query metnine göre
tekilleştirme) zararsızca elenir.

In [ ]:
import os, shutil, time

LOCAL_OUTPUT = "/kaggle/working/kaggle_judge_sonuc.csv"
MODEL_TAG = "gemma4:e4b"
CHUNK = 500  # her bu kadar YENİ satırda bir Kaggle Dataset'ine yedekle
TOTAL_TARGET = 143039  # TAM dosya

# Daha önceki bir yedekten devam ediyorsanız, o dataset'i "Add Input" ile ekleyip
# aşağıdaki satırı kendi yolunuzla açın:
# PREVIOUS_BACKUP = "/kaggle/input/kaggle-judge-output/kaggle_judge_sonuc.csv"
# if os.path.exists(PREVIOUS_BACKUP) and not os.path.exists(LOCAL_OUTPUT):
#     shutil.copy(PREVIOUS_BACKUP, LOCAL_OUTPUT)

print("TAM DOSYA (A→Z) + gemma4:e4b hakem koşusu başlıyor (duraklat+yedekle döngüsü)...")

while True:
    done_before = 0
    if os.path.exists(LOCAL_OUTPUT):
        with open(LOCAL_OUTPUT, newline="", encoding="utf-8") as f:
            done_before = sum(1 for _ in f) - 1

    if done_before >= TOTAL_TARGET:
        print(f"TAMAMLANDI: {done_before}/{TOTAL_TARGET}")
        break

    print(f"\n--- Bu turda hedef: +{CHUNK} satır (şu ana kadar: {done_before}/{TOTAL_TARGET}) ---")
    !python3 -m institution_resolver_v3.cli.main inventory-batch "{KAGGLE_INPUT_CSV_PATH}" --model "{MODEL_TAG}" --out "{LOCAL_OUTPUT}" --workers 2 --resume --limit {CHUNK}

    backup_to_kaggle_dataset(LOCAL_OUTPUT, message=f"progress after {done_before + CHUNK} rows (approx)")

    done_after = 0
    if os.path.exists(LOCAL_OUTPUT):
        with open(LOCAL_OUTPUT, newline="", encoding="utf-8") as f:
            done_after = sum(1 for _ in f) - 1
    if done_after == done_before:
        print("UYARI: bu turda hiç yeni satır işlenmedi - döngü durduruluyor (hata olabilir, logları kontrol edin).")
        break

print("\nDöngü sona erdi (oturum süresi/kota dolmuş olabilir - normal, sonraki oturumda --resume ile devam eder). Son durum Kaggle Dataset'inde yedekli.")

## 11) Manuel Ara Yedekleme (istediğiniz an)

10. hücre zaten periyodik yedekliyor, ama oturumu kapatmadan önce elle bir kez daha
tetiklemek isterseniz bu hücreyi kullanın.

In [ ]:
backup_to_kaggle_dataset("/kaggle/working/kaggle_judge_sonuc.csv", message="manuel yedek")